# Chart Transport MNIST

Flattened-MNIST chart transport with an anchored Gaussian scale-mixture prior,
`StackedResidualMLP` encoder/decoder/critic stacks of width `1024`, and `num_hidden_layers = 8`.
The notebook pins training to `cuda:1` and uses fixed validation monitor batches plus fixed prior latents
so the logged artifacts are comparable across stages.


In [1]:
import contextlib
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import torch
import torch.nn.functional as F
import wandb
from IPython.display import display
from lightning import Fabric
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.data.mnist.chart_transport import (
    ConstraintMonitorConfig,
    CriticMonitorConfig,
    IntegratedMonitorConfig,
    MNISTChartTransportTrainingConfig,
    MonitorConfig,
    get_canonical_chart_transport_config,
)
from src.data.mnist.data import MNISTDataConfig
from src.data.mnist.monitoring import (
    build_latent_grid,
    write_constraint_monitor_artifacts,
    write_critic_monitor_artifacts,
    write_integrated_monitor_artifacts,
)

torch.set_float32_matmul_precision("medium")


In [2]:
data_root = Path.cwd() / ".data"
latent_dimension = 64
num_hidden_layers = 10
experiment_name = f"{datetime.now():%m%d%H%M%S}-{uuid4().hex[:6]}"
wandb_project = "diffusive-latent-generation"

train_data_config = MNISTDataConfig.initialize(
    root=data_root,
    split="train",
    flatten=True,
    download=True,
)
val_data_config = MNISTDataConfig.initialize(
    root=data_root,
    split="val",
    flatten=True,
    download=True,
)

chart_transport_config = get_canonical_chart_transport_config(
    data_config=train_data_config,
    latent_dimension=latent_dimension,
    num_hidden_layers=num_hidden_layers,
)

training_config = MNISTChartTransportTrainingConfig.initialize(
    seed=0,
    train_batch_size=4096,
    eval_batch_size=4096,
    integrated_n_steps=1_000_000,
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/mnist_chart_transport")
        / experiment_name
    ),
)

monitor_config = MonitorConfig(
    constraint_monitor_config=ConstraintMonitorConfig(
        reconstruction_examples_per_class=3,
        latent_examples_per_class=256,
    ),
    critic_monitor_config=CriticMonitorConfig(
        sample_t_values=[0.03, 0.20, 0.50],
        num_contour_lines=10,
        dense_latent_examples_per_class=256,
        vector_latent_examples_per_class=16,
    ),
    integrated_monitor_config=IntegratedMonitorConfig(
        generated_grid_rows=6,
        generated_grid_cols=8,
    ),
    log_every_n_steps_constraint_pretrain=1000,
    log_every_n_steps_critic_pretrain=1000,
    log_every_n_steps_integrated=5_000,
)

display(chart_transport_config.visualize())
display(monitor_config.visualize())


In [3]:
fabric = Fabric(
    accelerator="cuda",
    devices=[1],
    precision="bf16-mixed",
)
fabric.launch()
fabric.seed_everything(training_config.seed)

device = fabric.device
device_type = device.type

architecture_config = chart_transport_config.architecture_config
chart_transport_model = architecture_config.get_model()
optimizer = architecture_config.get_optimizer(model=chart_transport_model)
chart_transport_model, optimizer = fabric.setup(chart_transport_model, optimizer)

encoder = chart_transport_model.encoder
decoder = chart_transport_model.decoder
critic = chart_transport_model.critic

prior_config = chart_transport_config.prior_config
constraint_config = chart_transport_config.loss_config.constraint_config
constraint_method = constraint_config.constraint_method
pretrain_config = chart_transport_config.loss_config.chart_pretrain_config
transport_config = chart_transport_config.loss_config.transport_config
critic_config = chart_transport_config.loss_config.critic_config
transport_t_min, transport_t_max = transport_config.t_range

data_dual = torch.zeros((), device=device)
prior_dual = torch.zeros((), device=device)

fabric.print(
    f"device={device}, precision=bf16-mixed, folder={training_config.folder}"
)

wandb_run = wandb.init(
    project=wandb_project,
    name=experiment_name,
    dir=str(training_config.folder),
    tags=["chart-transport", "mnist"],
    config={
        "notebook": "chart-transport-mnist.ipynb",
        "seed": training_config.seed,
        "train_batch_size": training_config.train_batch_size,
        "eval_batch_size": training_config.eval_batch_size,
        "integrated_n_steps": training_config.integrated_n_steps,
        "latent_dimension": latent_dimension,
        "num_hidden_layers": num_hidden_layers,
    },
)
for stage_name in ("pretrain", "critic_pretrain", "integrated"):
    wandb.define_metric(f"{stage_name}/local_step")
    wandb.define_metric(f"{stage_name}/*", step_metric=f"{stage_name}/local_step")


Using bfloat16 Automatic Mixed Precision (AMP)
Seed set to 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nlyu/.netrc.
wandb: Currently logged in as: lyuxingjian (lyuxingjian-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


device=cuda:1, precision=bf16-mixed, folder=/home/nlyu/Code/diffusive-latent-generation/artifacts/mnist_chart_transport/0328231406-ee05e6


In [4]:
@contextlib.contextmanager
def runtime_precision_context():
    with contextlib.ExitStack() as stack:
        if device_type == "cuda":
            stack.enter_context(torch.cuda.device(device))
        stack.enter_context(torch.device(str(device)))
        stack.enter_context(
            torch.autocast(device_type=device_type, dtype=torch.bfloat16)
        )
        yield


def should_log_monitor(
    *,
    step: int,
    total_steps: int,
    every_n_steps: int,
) -> bool:
    return step == 1 or step % every_n_steps == 0 or step == total_steps


def format_metrics_summary(
    metrics: dict[str, object],
) -> str:
    return ", ".join(
        f"{key}={value:.4f}"
        for key, value in metrics.items()
        if isinstance(value, float)
    )


def log_wandb_scalars(
    *,
    stage: str,
    step: int,
    metrics: dict[str, object],
) -> None:
    payload = {
        f"{stage}/{key}": value
        for key, value in metrics.items()
        if isinstance(value, float) and value == value
    }
    payload[f"{stage}/local_step"] = step
    wandb.log(payload)


def to_device_samples(
    samples: torch.Tensor,
) -> torch.Tensor:
    return samples.to(device=device, dtype=torch.float32, non_blocking=True)


def to_device_labels(
    labels: torch.Tensor,
) -> torch.Tensor:
    return labels.to(device=device, dtype=torch.long, non_blocking=True)


def sample_training_batch(
    *,
    batch_size: int,
) -> torch.Tensor:
    return to_device_samples(
        train_data_config.sample_unconditional(batch_size=batch_size)
    )


def sample_fixed_monitor_batch(
    *,
    data_config: MNISTDataConfig,
    batch_size_per_class: int,
    start_index: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    samples, labels = data_config.stratified_class_batch(
        batch_size_per_class=batch_size_per_class,
        start_index=start_index,
    )
    return to_device_samples(samples), to_device_labels(labels)


def monitor_output_folder(
    *,
    stage: str,
    step: int,
    artifact_group: str,
) -> Path:
    return training_config.folder / stage / str(step) / artifact_group


def sample_transport_times(
    *,
    batch_shape: tuple[int, ...],
) -> torch.Tensor:
    return transport_t_min + (transport_t_max - transport_t_min) * torch.rand(
        *batch_shape,
        device=device,
        dtype=torch.float32,
    )


def sample_stratified_transport_times(
    *,
    batch_size: int,
    num_time_samples: int,
) -> torch.Tensor:
    bin_edges = torch.linspace(
        transport_t_min,
        transport_t_max,
        num_time_samples + 1,
        device=device,
        dtype=torch.float32,
    )
    bin_starts = bin_edges[:-1]
    bin_widths = bin_edges[1:] - bin_edges[:-1]
    return bin_starts.unsqueeze(0) + bin_widths.unsqueeze(0) * torch.rand(
        batch_size,
        num_time_samples,
        device=device,
        dtype=torch.float32,
    )


def critic_score_from_noise_prediction(
    *,
    predicted_noise: torch.Tensor,
    t: torch.Tensor,
) -> torch.Tensor:
    return -predicted_noise / t.unsqueeze(-1)


def critic_spectrum_t_values() -> list[float]:
    return torch.linspace(
        transport_t_min,
        transport_t_max,
        19,
        dtype=torch.float32,
    ).tolist()


def sample_critic_snapshot(
    *,
    clean_latents: torch.Tensor,
    t_value: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    t = torch.full(
        (clean_latents.shape[0],),
        float(t_value),
        device=clean_latents.device,
        dtype=torch.float32,
    )
    noise = torch.randn_like(clean_latents)
    noised_latents = (
        (1.0 - t).unsqueeze(-1) * clean_latents + t.unsqueeze(-1) * noise
    )
    predicted_noise = critic(noised_latents, t).float()
    data_score = critic_score_from_noise_prediction(
        predicted_noise=predicted_noise,
        t=t,
    )
    return noised_latents.float(), data_score.float()


def estimate_clean_transport_field(
    *,
    clean_latents: torch.Tensor,
    t_values: list[float],
) -> torch.Tensor:
    if not t_values:
        raise ValueError("t_values must be non-empty")

    transport_field = torch.zeros_like(clean_latents, dtype=torch.float32)
    for t_value in t_values:
        t = torch.full(
            (clean_latents.shape[0],),
            float(t_value),
            device=clean_latents.device,
            dtype=torch.float32,
        )
        pullback_weight = transport_config.kl_weight_schedule.pullback_weight(
            t.float(),
        ).unsqueeze(-1)
        noise = torch.randn_like(clean_latents)

        def evaluate_with_noise(sampled_noise: torch.Tensor) -> torch.Tensor:
            noised_latents = (
                (1.0 - t).unsqueeze(-1) * clean_latents
                + t.unsqueeze(-1) * sampled_noise
            )
            predicted_noise = critic(noised_latents, t).float()
            prior_score = prior_config.analytic_score(
                noised_latents.float(),
                t.float(),
            ).float()
            return pullback_weight * (
                prior_score + predicted_noise / t.unsqueeze(-1)
            )

        transport_terms = evaluate_with_noise(noise)
        if transport_config.antipodal_estimate:
            transport_terms = 0.5 * (
                transport_terms + evaluate_with_noise(-noise)
            )
        transport_field = transport_field + transport_terms
    return transport_field / len(t_values)



def estimate_transport_targets(
    *,
    clean_latents: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    transport_source_latents = clean_latents.detach()
    transport_t = sample_stratified_transport_times(
        batch_size=transport_source_latents.shape[0],
        num_time_samples=transport_config.num_time_samples,
    )

    transport_source_latents = transport_source_latents.unsqueeze(1).expand(
        -1,
        transport_config.num_time_samples,
        -1,
    )
    transport_eps = torch.randn(
        transport_source_latents.shape[0],
        transport_source_latents.shape[1],
        transport_source_latents.shape[-1],
        device=device,
        dtype=transport_source_latents.dtype,
    )

    if transport_config.antipodal_estimate:
        transport_t = torch.cat([transport_t, transport_t], dim=1)
        transport_source_latents = transport_source_latents.repeat(1, 2, 1)
        transport_eps = torch.cat([transport_eps, -transport_eps], dim=1)

    transport_noised_latents = (
        (1.0 - transport_t).unsqueeze(-1) * transport_source_latents
        + transport_t.unsqueeze(-1) * transport_eps
    )
    flat_transport_noised_latents = transport_noised_latents.reshape(
        -1,
        transport_noised_latents.shape[-1],
    )
    flat_transport_t = transport_t.reshape(-1)

    transport_predicted_noise = critic(
        flat_transport_noised_latents,
        flat_transport_t,
    ).reshape_as(transport_noised_latents)
    transport_prior_score = prior_config.analytic_score(
        flat_transport_noised_latents.float(),
        flat_transport_t.float(),
    ).to(dtype=flat_transport_noised_latents.dtype).reshape_as(
        transport_noised_latents
    )
    transport_pullback_weight = (
        transport_config.kl_weight_schedule.pullback_weight(
            flat_transport_t.float(),
        )
        .to(dtype=flat_transport_noised_latents.dtype)
        .reshape_as(transport_t)
    )
    transport_field_terms = transport_pullback_weight.unsqueeze(-1) * (
        transport_prior_score
        + transport_predicted_noise / transport_t.unsqueeze(-1)
    )

    if transport_config.antipodal_estimate:
        midpoint = transport_config.num_time_samples
        transport_field_terms = 0.5 * (
            transport_field_terms[:, :midpoint]
            + transport_field_terms[:, midpoint:]
        )

    transport_field = transport_field_terms.mean(dim=1)
    transport_field_norm = transport_field.norm(dim=-1, keepdim=True)
    transport_step_size = torch.minimum(
        torch.full_like(
            transport_field_norm,
            transport_config.transport_step_size,
        ),
        torch.full_like(
            transport_field_norm,
            transport_config.transport_step_cap,
        )
        / transport_field_norm.clamp_min(1e-6),
    )
    transport_step = transport_step_size * transport_field
    transported_latents = clean_latents.detach() + transport_step
    return transported_latents, transport_field, transport_field_norm


def generated_statistics(
    *,
    samples: torch.Tensor,
) -> dict[str, float]:
    return {
        "generated_out_of_range_fraction": (
            (samples < 0.0) | (samples > 1.0)
        ).float().mean().detach().item(),
    }


fixed_reconstruction_samples, fixed_reconstruction_labels = sample_fixed_monitor_batch(
    data_config=val_data_config,
    batch_size_per_class=monitor_config.constraint_monitor_config.reconstruction_examples_per_class,
    start_index=0,
)
fixed_latent_samples, fixed_latent_labels = sample_fixed_monitor_batch(
    data_config=val_data_config,
    batch_size_per_class=monitor_config.constraint_monitor_config.latent_examples_per_class,
    start_index=monitor_config.constraint_monitor_config.reconstruction_examples_per_class,
)
fixed_dense_samples, fixed_dense_labels = sample_fixed_monitor_batch(
    data_config=val_data_config,
    batch_size_per_class=monitor_config.critic_monitor_config.dense_latent_examples_per_class,
    start_index=0,
)
fixed_vector_samples, fixed_vector_labels = sample_fixed_monitor_batch(
    data_config=val_data_config,
    batch_size_per_class=monitor_config.critic_monitor_config.vector_latent_examples_per_class,
    start_index=monitor_config.critic_monitor_config.dense_latent_examples_per_class,
)
fixed_generated_prior = prior_config.sample(
    batch_size=(
        monitor_config.integrated_monitor_config.generated_grid_rows
        * monitor_config.integrated_monitor_config.generated_grid_cols
    )
).to(device=device, dtype=torch.float32)


## Pretrain


In [5]:
latest_pretrain_metrics = None
encoder.train()
decoder.train()
critic.eval()

pretrain_progress = tqdm(
    range(1, chart_transport_config.scheduling_config.pretrain_chart_n_steps + 1),
    desc="pretrain",
)
for step in pretrain_progress:
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        data_batch = sample_training_batch(
            batch_size=training_config.train_batch_size,
        )
        prior_batch = prior_config.sample(
            batch_size=training_config.train_batch_size,
        ).to(device=device, dtype=torch.float32)

        data_latents = encoder(data_batch)
        data_reconstruction = decoder(data_latents)
        prior_reconstruction = decoder(prior_batch)
        prior_latents = encoder(prior_reconstruction)

        data_cycle_loss = F.huber_loss(
            data_reconstruction,
            data_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        prior_cycle_loss = F.huber_loss(
            prior_latents,
            prior_batch,
            delta=constraint_config.huber_delta,
            reduction="mean",
        )
        constraint_loss = data_cycle_loss + prior_cycle_loss

        zero_mean_loss = F.huber_loss(
            data_latents.mean(),
            torch.zeros((), device=device, dtype=data_latents.dtype),
            delta=1.0,
            reduction="mean",
        )
        latent_norms = data_latents.norm(dim=-1)
        latent_norm_loss = F.huber_loss(
            latent_norms,
            torch.zeros_like(latent_norms),
            delta=pretrain_config.latent_norm_delta,
            reduction="mean",
        )
        chart_loss = constraint_loss
        chart_loss = chart_loss + pretrain_config.zero_mean_weight * zero_mean_loss
        chart_loss = chart_loss + pretrain_config.latent_norm_weight * latent_norm_loss

    fabric.backward(chart_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "pretrain",
        "step": step,
        "chart_loss": chart_loss.detach().item(),
        "data_cycle_loss": data_cycle_loss.detach().item(),
        "prior_cycle_loss": prior_cycle_loss.detach().item(),
        "zero_mean_loss": zero_mean_loss.detach().item(),
        "latent_norm_loss": latent_norm_loss.detach().item(),
    }
    latest_pretrain_metrics = metrics
    log_wandb_scalars(stage="pretrain", step=step, metrics=metrics)
    pretrain_progress.set_postfix(
        chart_loss=f"{metrics['chart_loss']:.4f}",
        data_cycle=f"{metrics['data_cycle_loss']:.4f}",
        prior_cycle=f"{metrics['prior_cycle_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=chart_transport_config.scheduling_config.pretrain_chart_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_constraint_pretrain,
    ):
        fabric.print(
            f"[pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_chart_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        decoder_was_training = decoder.training
        encoder.eval()
        decoder.eval()

        with torch.no_grad():
            with runtime_precision_context():
                fixed_reconstructions = decoder(encoder(fixed_reconstruction_samples))
                fixed_latent_values = encoder(fixed_latent_samples)

        if encoder_was_training:
            encoder.train()
        if decoder_was_training:
            decoder.train()

        write_constraint_monitor_artifacts(
            data_config=val_data_config,
            output_folder=monitor_output_folder(
                stage="pretrain",
                step=step,
                artifact_group="constraint",
            ),
            reconstruction_samples=fixed_reconstruction_samples.detach().cpu().float(),
            reconstruction_labels=fixed_reconstruction_labels.detach().cpu().long(),
            reconstructions=fixed_reconstructions.detach().cpu().float(),
            latent_values=fixed_latent_values.detach().cpu().float(),
            latent_labels=fixed_latent_labels.detach().cpu().long(),
            examples_per_class=monitor_config.constraint_monitor_config.reconstruction_examples_per_class,
        )

latest_pretrain_metrics


pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[pretrain] step 1/2000: chart_loss=0.7164, data_cycle_loss=0.1353, prior_cycle_loss=0.5291, zero_mean_loss=0.0012, latent_norm_loss=5.2007
[pretrain] step 1000/2000: chart_loss=0.0076, data_cycle_loss=0.0063, prior_cycle_loss=0.0003, zero_mean_loss=0.0000, latent_norm_loss=0.1015
[pretrain] step 2000/2000: chart_loss=0.0053, data_cycle_loss=0.0046, prior_cycle_loss=0.0002, zero_mean_loss=0.0000, latent_norm_loss=0.0532


{'stage': 'pretrain',
 'step': 2000,
 'chart_loss': 0.005276510026305914,
 'data_cycle_loss': 0.0045563699677586555,
 'prior_cycle_loss': 0.00018786455621011555,
 'zero_mean_loss': 1.0477378964424133e-07,
 'latent_norm_loss': 0.05322744697332382}

## Train noise critic


In [6]:
latest_critic_pretrain_metrics = None
encoder.eval()
decoder.eval()
critic.train()

critic_pretrain_progress = tqdm(
    range(1, chart_transport_config.scheduling_config.pretrain_critic_n_steps + 1),
    desc="critic_pretrain",
)
for step in critic_pretrain_progress:
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        data_batch = sample_training_batch(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            data_latents = encoder(data_batch)

        t = sample_transport_times(
            batch_shape=(data_latents.shape[0],),
        )
        eps = torch.randn_like(data_latents)
        noised_latents = (
            (1.0 - t).unsqueeze(-1) * data_latents + t.unsqueeze(-1) * eps
        )
        predicted_noise = critic(noised_latents, t)
        critic_loss = critic_config.loss_weight * F.mse_loss(predicted_noise, eps)

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "critic_pretrain",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
    }
    latest_critic_pretrain_metrics = metrics
    log_wandb_scalars(stage="critic_pretrain", step=step, metrics=metrics)
    critic_pretrain_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=chart_transport_config.scheduling_config.pretrain_critic_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_critic_pretrain,
    ):
        fabric.print(
            f"[critic_pretrain] step {step}/{chart_transport_config.scheduling_config.pretrain_critic_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        critic_was_training = critic.training
        encoder.eval()
        critic.eval()

        with torch.no_grad():
            with runtime_precision_context():
                dense_clean_latents = encoder(fixed_dense_samples).float()
                vector_clean_latents = encoder(fixed_vector_samples).float()

                score_snapshots = []
                for t_value in monitor_config.critic_monitor_config.sample_t_values:
                    cloud_latents, _ = sample_critic_snapshot(
                        clean_latents=dense_clean_latents,
                        t_value=t_value,
                    )
                    arrow_latents, data_score = sample_critic_snapshot(
                        clean_latents=vector_clean_latents,
                        t_value=t_value,
                    )
                    score_snapshots.append(
                        (
                            t_value,
                            cloud_latents.detach().cpu().float(),
                            arrow_latents.detach().cpu().float(),
                            data_score.detach().cpu().float(),
                        )
                    )

                spectrum_t_values = critic_spectrum_t_values()
                transport_field = estimate_clean_transport_field(
                    clean_latents=vector_clean_latents,
                    t_values=spectrum_t_values,
                )
                transport_grid_points, transport_grid_xs, transport_grid_ys = build_latent_grid(
                    reference_points=dense_clean_latents,
                    resolution=31,
                )
                transport_grid_field = estimate_clean_transport_field(
                    clean_latents=transport_grid_points,
                    t_values=spectrum_t_values,
                )
                transport_grid_projection = transport_grid_field.float()
                transport_grid_projection = transport_grid_projection.reshape(
                    transport_grid_ys.shape[0],
                    transport_grid_xs.shape[0],
                    -1,
                ).norm(dim=-1)

        if encoder_was_training:
            encoder.train()
        if critic_was_training:
            critic.train()

        write_critic_monitor_artifacts(
            output_folder=monitor_output_folder(
                stage="critic_pretrain",
                step=step,
                artifact_group="critic",
            ),
            dense_clean_latents=dense_clean_latents.detach().cpu().float(),
            dense_labels=fixed_dense_labels.detach().cpu().long(),
            vector_clean_latents=vector_clean_latents.detach().cpu().float(),
            vector_labels=fixed_vector_labels.detach().cpu().long(),
            score_snapshots=score_snapshots,
            transport_field=transport_field.detach().cpu().float(),
            transport_grid_xs=transport_grid_xs.detach().cpu().float(),
            transport_grid_ys=transport_grid_ys.detach().cpu().float(),
            transport_grid_projection=transport_grid_projection.detach().cpu().float(),
            num_contour_lines=monitor_config.critic_monitor_config.num_contour_lines,
        )

latest_critic_pretrain_metrics


critic_pretrain:   0%|          | 0/2000 [00:00<?, ?it/s]

[critic_pretrain] step 1/2000: critic_loss=1.1734
[critic_pretrain] step 1000/2000: critic_loss=0.0293
[critic_pretrain] step 2000/2000: critic_loss=0.0287


{'stage': 'critic_pretrain', 'step': 2000, 'critic_loss': 0.028748704120516777}

## Integrated training


In [ ]:
latest_integrated_metrics = None
critic_steps_per_chart_step = max(
    1,
    chart_transport_config.scheduling_config.update_chart_every_n_critic_steps,
)
latest_chart_metrics = {
    "chart_loss": float("nan"),
    "encoder_transport_loss": float("nan"),
    "decoder_transport_loss": float("nan"),
    "transport_field_norm": float("nan"),
    "generated_out_of_range_fraction": float("nan"),
    "data_cycle_loss": float("nan"),
    "prior_cycle_loss": float("nan"),
}

integrated_progress = tqdm(
    range(1, training_config.integrated_n_steps + 1),
    desc="integrated",
)
for step in integrated_progress:
    encoder.eval()
    decoder.eval()
    critic.train()
    optimizer.zero_grad(set_to_none=True)

    with runtime_precision_context():
        critic_data_batch = sample_training_batch(
            batch_size=training_config.train_batch_size,
        )
        with torch.no_grad():
            critic_data_latents = encoder(critic_data_batch)

        critic_t = sample_transport_times(
            batch_shape=(critic_data_latents.shape[0],),
        )
        critic_eps = torch.randn_like(critic_data_latents)
        critic_noised_latents = (
            (1.0 - critic_t).unsqueeze(-1) * critic_data_latents
            + critic_t.unsqueeze(-1) * critic_eps
        )
        critic_predicted_noise = critic(critic_noised_latents, critic_t)
        critic_loss = critic_config.loss_weight * F.mse_loss(
            critic_predicted_noise,
            critic_eps,
        )

    fabric.backward(critic_loss)
    fabric.clip_gradients(
        chart_transport_model,
        optimizer,
        max_norm=architecture_config.grad_clip_norm,
    )
    optimizer.step()

    metrics = {
        "stage": "integrated",
        "step": step,
        "critic_loss": critic_loss.detach().item(),
        **latest_chart_metrics,
        "data_dual": data_dual.detach().item(),
        "prior_dual": prior_dual.detach().item(),
    }
    wandb_metrics = {
        "critic_loss": critic_loss.detach().item(),
        "data_dual": data_dual.detach().item(),
        "prior_dual": prior_dual.detach().item(),
    }

    if step == 1 or step % critic_steps_per_chart_step == 0:
        encoder.train()
        decoder.train()
        optimizer.zero_grad(set_to_none=True)

        with runtime_precision_context():
            chart_data_batch = sample_training_batch(
                batch_size=training_config.train_batch_size,
            )
            chart_prior_batch = prior_config.sample(
                batch_size=training_config.train_batch_size,
            ).to(device=device, dtype=torch.float32)

            chart_data_latents = encoder(chart_data_batch)
            chart_data_reconstruction = decoder(chart_data_latents)
            chart_prior_reconstruction = decoder(chart_prior_batch)
            chart_prior_latents = encoder(chart_prior_reconstruction)

            data_cycle_loss = F.huber_loss(
                chart_data_reconstruction,
                chart_data_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )
            prior_cycle_loss = F.huber_loss(
                chart_prior_latents,
                chart_prior_batch,
                delta=constraint_config.huber_delta,
                reduction="mean",
            )

            if isinstance(constraint_method, LossConstraintConfig):
                constraint_loss = (
                    constraint_method.data_loss_weight * data_cycle_loss
                    + constraint_method.prior_loss_weight * prior_cycle_loss
                )
            else:
                constraint_loss = data_cycle_loss + prior_cycle_loss
                constraint_loss = constraint_loss + data_dual * (
                    data_cycle_loss - constraint_method.data_constraint_budget
                )
                constraint_loss = constraint_loss + prior_dual * (
                    prior_cycle_loss - constraint_method.prior_constraint_budget
                )

            with torch.no_grad():
                transported_latents, transport_field, transport_field_norm = estimate_transport_targets(
                    clean_latents=chart_data_latents,
                )

            encoder_transport_loss = F.mse_loss(
                chart_data_latents,
                transported_latents.detach(),
            )
            decoder_transport_loss = F.mse_loss(
                decoder(transported_latents),
                chart_data_batch.detach(),
            )
            chart_loss = constraint_loss
            chart_loss = chart_loss + (
                transport_config.encoder_transport_weight * encoder_transport_loss
            )
            chart_loss = chart_loss + (
                transport_config.decoder_transport_weight * decoder_transport_loss
            )

        fabric.backward(chart_loss)
        fabric.clip_gradients(
            chart_transport_model,
            optimizer,
            max_norm=architecture_config.grad_clip_norm,
        )
        optimizer.step()

        if isinstance(constraint_method, LagrangianConstraintConfig):
            data_dual = (
                data_dual
                + constraint_method.dual_variable_lr
                * (
                    data_cycle_loss.detach()
                    - constraint_method.data_constraint_budget
                )
            ).clamp_min(0.0)
            prior_dual = (
                prior_dual
                + constraint_method.dual_variable_lr
                * (
                    prior_cycle_loss.detach()
                    - constraint_method.prior_constraint_budget
                )
            ).clamp_min(0.0)

        latest_chart_metrics = {
            "chart_loss": chart_loss.detach().item(),
            "encoder_transport_loss": encoder_transport_loss.detach().item(),
            "decoder_transport_loss": decoder_transport_loss.detach().item(),
            "transport_field_norm": transport_field_norm.mean().detach().item(),
            "data_cycle_loss": data_cycle_loss.detach().item(),
            "prior_cycle_loss": prior_cycle_loss.detach().item(),
            **generated_statistics(samples=chart_prior_reconstruction.float()),
        }
        metrics.update(latest_chart_metrics)
        metrics["data_dual"] = data_dual.detach().item()
        metrics["prior_dual"] = prior_dual.detach().item()
        wandb_metrics.update(latest_chart_metrics)
        wandb_metrics["data_dual"] = data_dual.detach().item()
        wandb_metrics["prior_dual"] = prior_dual.detach().item()

    latest_integrated_metrics = metrics
    log_wandb_scalars(stage="integrated", step=step, metrics=wandb_metrics)
    integrated_progress.set_postfix(
        critic_loss=f"{metrics['critic_loss']:.4f}",
        chart_loss=f"{metrics['chart_loss']:.4f}",
    )

    if should_log_monitor(
        step=step,
        total_steps=training_config.integrated_n_steps,
        every_n_steps=monitor_config.log_every_n_steps_integrated,
    ):
        fabric.print(
            f"[integrated] step {step}/{training_config.integrated_n_steps}: {format_metrics_summary(metrics)}"
        )

        encoder_was_training = encoder.training
        decoder_was_training = decoder.training
        critic_was_training = critic.training
        encoder.eval()
        decoder.eval()
        critic.eval()

        with torch.no_grad():
            with runtime_precision_context():
                fixed_reconstructions = decoder(encoder(fixed_reconstruction_samples))
                fixed_latent_values = encoder(fixed_latent_samples)

                dense_clean_latents = encoder(fixed_dense_samples).float()
                vector_clean_latents = encoder(fixed_vector_samples).float()

                score_snapshots = []
                for t_value in monitor_config.critic_monitor_config.sample_t_values:
                    cloud_latents, _ = sample_critic_snapshot(
                        clean_latents=dense_clean_latents,
                        t_value=t_value,
                    )
                    arrow_latents, data_score = sample_critic_snapshot(
                        clean_latents=vector_clean_latents,
                        t_value=t_value,
                    )
                    score_snapshots.append(
                        (
                            t_value,
                            cloud_latents.detach().cpu().float(),
                            arrow_latents.detach().cpu().float(),
                            data_score.detach().cpu().float(),
                        )
                    )

                spectrum_t_values = critic_spectrum_t_values()
                transport_field = estimate_clean_transport_field(
                    clean_latents=vector_clean_latents,
                    t_values=spectrum_t_values,
                )
                transport_grid_points, transport_grid_xs, transport_grid_ys = build_latent_grid(
                    reference_points=dense_clean_latents,
                    resolution=31,
                )
                transport_grid_field = estimate_clean_transport_field(
                    clean_latents=transport_grid_points,
                    t_values=spectrum_t_values,
                )
                transport_grid_projection = transport_grid_field.float()
                transport_grid_projection = transport_grid_projection.reshape(
                    transport_grid_ys.shape[0],
                    transport_grid_xs.shape[0],
                    -1,
                ).norm(dim=-1)

                generated_samples = decoder(fixed_generated_prior)

        if encoder_was_training:
            encoder.train()
        if decoder_was_training:
            decoder.train()
        if critic_was_training:
            critic.train()

        write_constraint_monitor_artifacts(
            data_config=val_data_config,
            output_folder=monitor_output_folder(
                stage="integrated",
                step=step,
                artifact_group="constraint",
            ),
            reconstruction_samples=fixed_reconstruction_samples.detach().cpu().float(),
            reconstruction_labels=fixed_reconstruction_labels.detach().cpu().long(),
            reconstructions=fixed_reconstructions.detach().cpu().float(),
            latent_values=fixed_latent_values.detach().cpu().float(),
            latent_labels=fixed_latent_labels.detach().cpu().long(),
            examples_per_class=monitor_config.constraint_monitor_config.reconstruction_examples_per_class,
        )
        write_critic_monitor_artifacts(
            output_folder=monitor_output_folder(
                stage="integrated",
                step=step,
                artifact_group="critic",
            ),
            dense_clean_latents=dense_clean_latents.detach().cpu().float(),
            dense_labels=fixed_dense_labels.detach().cpu().long(),
            vector_clean_latents=vector_clean_latents.detach().cpu().float(),
            vector_labels=fixed_vector_labels.detach().cpu().long(),
            score_snapshots=score_snapshots,
            transport_field=transport_field.detach().cpu().float(),
            transport_grid_xs=transport_grid_xs.detach().cpu().float(),
            transport_grid_ys=transport_grid_ys.detach().cpu().float(),
            transport_grid_projection=transport_grid_projection.detach().cpu().float(),
            num_contour_lines=monitor_config.critic_monitor_config.num_contour_lines,
        )
        write_integrated_monitor_artifacts(
            data_config=val_data_config,
            output_folder=monitor_output_folder(
                stage="integrated",
                step=step,
                artifact_group="summary",
            ),
            generated_samples=generated_samples.detach().cpu().float(),
            generated_grid_rows=monitor_config.integrated_monitor_config.generated_grid_rows,
            generated_grid_cols=monitor_config.integrated_monitor_config.generated_grid_cols,
        )

wandb.finish()
latest_integrated_metrics


integrated:   0%|          | 0/1000000 [00:00<?, ?it/s]

[integrated] step 1/1000000: critic_loss=0.0298, chart_loss=0.0136, encoder_transport_loss=0.0000, decoder_transport_loss=0.0088, transport_field_norm=6.2516, generated_pixel_mean=0.0265, generated_pixel_std=1.0080, generated_out_of_range_fraction=0.6522, data_cycle_loss=0.0046, prior_cycle_loss=0.0002, data_dual=0.0000, prior_dual=0.0000
[integrated] step 5000/1000000: critic_loss=0.2170, chart_loss=0.0076, encoder_transport_loss=0.0000, decoder_transport_loss=0.0062, transport_field_norm=7.6928, data_cycle_loss=0.0031, prior_cycle_loss=0.0006, generated_pixel_mean=0.0001, generated_pixel_std=0.6648, generated_out_of_range_fraction=0.5604, data_dual=0.1648, prior_dual=0.1176
[integrated] step 10000/1000000: critic_loss=0.2165, chart_loss=0.0055, encoder_transport_loss=0.0000, decoder_transport_loss=0.0059, transport_field_norm=7.7459, data_cycle_loss=0.0029, prior_cycle_loss=0.0007, generated_pixel_mean=0.0523, generated_pixel_std=0.8504, generated_out_of_range_fraction=0.5883, data_d